<a href="https://colab.research.google.com/github/bmulaw/GraphSystems/blob/main/GraphSystems_AirlineNetwork_GraphFrames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apache Spark GraphFrames: guided notebook (2026-2027)

This notebook follows the **GraphFrames** part of the course slides, section by section.
Each section has the same structure:

1. **Slides**: the slide titles to look at in the deck.
2. **Demo**: the code of the slide, to run and read. Each code cell starts with a `# Goal:` line and gives **the same output as the slide**.
3. **Your turn**: a short exercise that uses the same notions on a second graph, with hints and the **expected result** so you can check yourself.

**How to work**

* Run the cells **in order** with `Shift+Enter`. If something breaks, use *Runtime > Restart session and run all*.
* In the *Your turn* cells, replace the comments by your code. The commented lines give a skeleton: fill the `...`.
* Exercises marked **(bonus)** are optional. The last section is a **challenge** that is not in the slides.
* A `show()` without `orderBy` can print the rows in another order than on the slide: Spark guarantees no order. The rows are the same.
* A solutions notebook is given after the class.

**Data**: nothing to upload, the two graphs are written in the notebook.
* **The course graph** (the demos): 7 users `a`-`g` with their age, 8 relationships `friend` / `follow`. Gabby (`g`) has no relationship.
* **The airline network** (the *Your turn* exercises): 11 airports and 13 flights, built in *Your turn 1*.

## 0. Setup

**Slides**: *Installing GraphFrames 0.12 in Colab*

`graphframes-py` is only the Python client. The Scala code of GraphFrames is a JAR that Spark downloads from Maven Central the first time it starts (cell *Start Spark with GraphFrames*, below): Colab needs internet access, which it has.
Colab already has Java 21 (Spark 4 needs Java 17 or more) and an older PySpark. We install the versions of the course (PySpark 4.3.0, GraphFrames 0.12.3).
We also remove `dataproc-spark-connect`, a Google Cloud package tied to the old PySpark: it is not used here and would only print version conflicts.
You can ignore the message *"you may need to restart the kernel"*: PySpark has not been imported yet.

In [ ]:
# Goal: install PySpark 4.3.0 and graphframes-py 0.12.3 (about 1 minute)
%pip uninstall -y -q dataproc-spark-connect
%pip install -q pyspark==4.3.0 graphframes-py==0.12.3

In [ ]:
# Goal: check the versions (expected: PySpark 4.3.0, graphframes-py 0.12.3, Java 17 or 21)
import sys, pyspark
from importlib.metadata import version
print("Python        ", sys.version.split()[0])
print("PySpark       ", pyspark.__version__)
print("graphframes-py", version("graphframes-py"))
!java -version 2>&1 | head -1

**Start Spark with GraphFrames.** This cell is the code of the slide (`spark.jars.packages` loads the GraphFrames JAR), plus three lines that are **not on the slide**:

* `spark.sql.shuffle.partitions`: number of partitions after a shuffle (`groupBy`, join). The default, 200, is made for clusters. 4 is enough for our small graphs and much faster. It also gives the same component ids and labels as on the slides.
* `spark.ui.showConsoleProgress`: hides the `[Stage 3:>   (0 + 1) / 1]` progress bars in the outputs.
* `setLogLevel("ERROR")`: hides the INFO and WARN messages.

The first start downloads the JAR: it takes a few seconds more.

In [ ]:
# Goal: a SparkSession that can load GraphFrames
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from graphframes import GraphFrame
pkg = "io.graphframes:graphframes-spark4_2.13:0.12.3"
spark = (SparkSession.builder
         .master("local[*]")
         .config("spark.jars.packages", pkg)
         .config("spark.sql.shuffle.partitions", "4")       # not on the slide: small graphs
         .config("spark.ui.showConsoleProgress", "false")   # not on the slide: no progress bars
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")                     # not on the slide: hide INFO and WARN
spark.version      # '4.3.0'

## 1. Building a GraphFrame

**Slides**: *Building a GraphFrame*

A GraphFrame is built from **two DataFrames**:
* the **vertices**, with a column `id` (unique). The other columns are attributes of the vertex.
* the **edges**, with the columns `src` and `dst` (ids of vertices). The other columns are attributes of the edge.

This is the **course graph**: 7 users and 8 relationships. It is used in all the demos.

In [ ]:
# Goal: build the course graph (7 users, 8 relationships) from two DataFrames
v = spark.createDataFrame([
    ("a", "Alice", 34), ("b", "Bob", 36), ("c", "Charlie", 30), ("d", "David", 29),
    ("e", "Esther", 32), ("f", "Fanny", 36), ("g", "Gabby", 60)],
    ["id", "name", "age"])
e = spark.createDataFrame([
    ("a", "b", "friend"), ("b", "c", "follow"), ("c", "b", "follow"),
    ("f", "c", "follow"), ("e", "f", "follow"), ("e", "d", "friend"),
    ("d", "a", "friend"), ("a", "e", "friend")],
    ["src", "dst", "relationship"])
g = GraphFrame(v, e)
print(g)
print(g.vertices.count(), g.edges.count())

### Your turn 1: the airline network

The exercises use a second graph, an **airline network**: the vertices are airports, the edges are flights.
`pax` = passengers per year, in millions (invented figures, for the exercises). `km` = length of the flight.
Run the next cell: it only defines the data as two Python lists.

In [ ]:
# Goal: the data of the airline network (11 airports, 13 flights)
airports = [("CDG", "Paris", "FR", 76), ("LHR", "London", "GB", 80), ("FRA", "Frankfurt", "DE", 70),
            ("MAD", "Madrid", "ES", 60), ("LIS", "Lisbon", "PT", 33), ("JFK", "New York", "US", 62),
            ("BOS", "Boston", "US", 42), ("ORD", "Chicago", "US", 80), ("NRT", "Tokyo", "JP", 40),
            ("KIX", "Osaka", "JP", 30), ("PPT", "Papeete", "PF", 1)]
flights = [("CDG", "LHR", "AF", 344), ("LHR", "CDG", "BA", 344), ("CDG", "FRA", "AF", 479),
           ("FRA", "LHR", "LH", 637), ("LHR", "JFK", "BA", 5540), ("JFK", "BOS", "AA", 300),
           ("BOS", "ORD", "AA", 1380), ("ORD", "JFK", "AA", 1190), ("JFK", "CDG", "AF", 5830),
           ("MAD", "CDG", "IB", 1050), ("LIS", "MAD", "TP", 500), ("NRT", "KIX", "NH", 400),
           ("KIX", "NRT", "JL", 400)]

**Goal**: build the GraphFrame `air` from these two lists. The airports have the columns `id` (the airport code), `city`, `country`, `pax`. The flights have the columns `src`, `dst`, `airline`, `km`. Print the graph, then its number of airports and of flights.

*Hint*: `spark.createDataFrame(list, [column names])` twice, then `GraphFrame(vertices, edges)`, `count()`.

*Expected*: `GraphFrame(v:DataFrame[id: string, city: string, country: string, pax: bigint], e:DataFrame[src: string, dst: string, airline: string, km: bigint])`, then `11 13`.

In [ ]:
# YOUR CODE HERE
# v_air = spark.createDataFrame(airports, [...])
# e_air = spark.createDataFrame(flights, [...])
# air = GraphFrame(..., ...)
# print(air)
# print(...)

**Checkpoint**: run this cell even if your answer works. All the next exercises need exactly this graph `air`.

In [ ]:
# Goal: checkpoint, the airline network used by all the next exercises
air = GraphFrame(spark.createDataFrame(airports, ["id", "city", "country", "pax"]),
                 spark.createDataFrame(flights, ["src", "dst", "airline", "km"]))
print(air.vertices.count(), air.edges.count())

## 2. Vertices and edges DataFrames

**Slides**: *Vertices and edges DataFrames*

`g.vertices` and `g.edges` are ordinary DataFrames: everything you know about DataFrames works on them (`filter`, `groupBy`, `join`, `show`, SQL...).
Gabby appears in no edge: she is an **isolated vertex**.

In [ ]:
# Goal: look at the two tables of the graph
g.vertices.show()
g.edges.show()

### Your turn 2

**Goal**: how many flights does each airline operate? Show the 3 airlines with the most flights (ties broken by airline code). Then print the schema of the edges: what is the type of `km`?

*Hint*: `air.edges.groupBy("airline").count()`, `orderBy(F.desc("count"), "airline")`, `printSchema()`.

*Expected*: `AA 3`, `AF 3`, `BA 2`. The type of `km` is `long`.

In [ ]:
# YOUR CODE HERE
# air.edges.groupBy(...).count().orderBy(...).show(3)
# air.edges.printSchema()

## 3. Triplets

**Slides**: *Triplets: the join of the two tables*

A **triplet** = a source vertex, an edge and a destination vertex. `g.triplets` is the edges table joined with the vertices table twice.
Its three columns are **structs**: `src.age`, `edge.relationship`, `dst.name`.

In [ ]:
# Goal: see edges together with their endpoints
g.triplets.show(truncate=False)
g.triplets.filter("src.age < dst.age").select("src.id", "dst.id").show()

### Your turn 3

**Goal**: the flights longer than 1,000 km, with the city of departure, the city of arrival and the length. Sort from the longest to the shortest.

*Hint*: `air.triplets.filter("edge.km > 1000")`, then `select(F.col("src.city").alias("from_city"), ...)`: both cities are called `city`, so give them an alias.

*Expected*: 5 flights: New York to Paris 5830, London to New York 5540, Boston to Chicago 1380, Chicago to New York 1190, Madrid to Paris 1050.

In [ ]:
# YOUR CODE HERE
# air.triplets.filter(...) \
#     .select(F.col("src.city").alias("from_city"), ..., F.col("edge.km").alias("km")) \
#     .orderBy(...).show(truncate=False)

## 4. Vertex degrees

**Slides**: *Vertex degrees*

`g.inDegrees`, `g.outDegrees`, `g.degrees` (in + out) count the edges of each vertex.
They are computed with a `groupBy` on the edges, so a vertex with **no edge is missing** (Gabby, here). A join with `g.vertices` gives them back.

In [ ]:
# Goal: count edges per vertex
g.inDegrees.orderBy("id").show()
g.outDegrees.orderBy("id").show()
g.degrees.orderBy("id").show()

### Your turn 4

**Goal**: which airport has the highest degree, and what is it? Which airports have **no flight at all** (they are missing from `air.degrees`)? Show their `id` and `city`.

*Hint*: `air.degrees.orderBy(F.desc("degree"), "id")`. For the missing airports: a `left_anti` join of `air.vertices` with `air.degrees` on `id` (the rows of the left table with no match).

*Expected*: `CDG` has degree 5. The only airport without flight is `PPT` (Papeete).

In [ ]:
# YOUR CODE HERE
# air.degrees.orderBy(...).show(1)
# air.vertices.join(air.degrees, "id", ...).select("id", "city").show()

## 5. Motif finding

**Slides**: *Motif finding (1): a pattern is a small graph*, *Motif finding (2): example*

`g.find(pattern)` looks for a small graph, written in a Cypher-like ASCII syntax, inside the big one. The result is a DataFrame with **one row per match**.
* `(a)-[e]->(b)`: one edge `e` from `a` to `b`. `(a)`, `[e]` are names: each becomes a struct column (`a.name`, `e.relationship`).
* `(a)-[e1]->(b); (b)-[e2]->(c)`: several edges separated by `;`. A name used twice is the **same** vertex.
* `!(a)-[]->(c)`: negation, this edge must be **absent**. `()` and `[]` are anonymous.
* Two names are not forced to be different vertices: add `filter("a.id != c.id")` when needed.

In [ ]:
# Goal: friends of friends who are not already connected
fof = (g.find("(x)-[]->(y); (y)-[]->(z); !(x)-[]->(z)")
       .filter("x.id != z.id"))
fof.select(F.col("x.id").alias("x"), F.col("y.id").alias("y"),
           F.col("z.id").alias("z")).show()

### Your turn 5

**Goal**: find the pairs of airports (`x`, `z`) with a connection through one airport `y` (`x` to `y` to `z`) but **no direct flight** `x` to `z`, and `x` different from `z`. How many matches are there? Which airports can Lisbon (`LIS`) reach in this way?

*Hint*: Same pattern as the slide: `"(x)-[]->(y); (y)-[]->(z); !(x)-[]->(z)"`, then `filter("x.id != z.id")`, `count()`. To show the ids: `F.col("x.id").alias("x")`.

*Expected*: 14 matches. From Lisbon: only `LIS`, `MAD`, `CDG` (Lisbon to Madrid to Paris).

In [ ]:
# YOUR CODE HERE
# fof_air = air.find("...").filter("x.id != z.id")
# print(fof_air.count())
# fof_air.filter("x.id = 'LIS'").select(F.col("x.id").alias("x"), ..., ...).show()

## 6. Motif finding: the 0.10+ syntax

**Slides**: *Motif finding (3): 0.10+ syntax*

Since GraphFrames 0.10 the patterns have more operators:
* `(a)<-[e]->(b)`: **bidirectional**, edges in both directions between `a` and `b` (a mutual relationship). Both orientations are returned.
* `(a)-[e*2]->(c)`: a path of **exactly 2** edges. `e*1..2`: a path of 1 **or** 2 edges (variable length).
* `(a)-[e]-(b)`: **undirected**, each edge is matched in both directions.

In [ ]:
# Goal: mutual relationships and two-hop paths with the 0.10+ pattern syntax
g.find("(a)<-[e]->(b)").select("a.id", "b.id").show()      # bidirectional
two = g.find("(a)-[e*2]->(c)")                             # fixed length 2
two.select("a.id", "c.id").orderBy("a.id", "c.id").show()
print(two.count())
print(g.find("(a)-[e*1..2]->(c)").count())                 # variable length 1..2

### Your turn 6

**Goal**: (a) The pairs of airports with a flight in **both** directions, one row per pair (keep `a.id < b.id`). (b) The airports reached in exactly 2 flights from New York (`JFK`). (c) How many paths of exactly 2 flights are there in the network, and how many of 1 or 2 flights?

*Hint*: Bidirectional: `"(a)<-[e]->(b)"`. Two flights: `"(a)-[e*2]->(c)"`. Variable length: `"(a)-[e*1..2]->(c)"`.

*Expected*: (a) `CDG`-`LHR` and `KIX`-`NRT`. (b) `FRA`, `LHR`, `ORD`. (c) 20 paths of exactly 2 flights, 33 of 1 or 2 flights (33 = 13 + 20: the network has 13 flights).

In [ ]:
# YOUR CODE HERE
# air.find("...").filter("a.id < b.id").select("a.id", "b.id").show()
# two_air = air.find("...")
# two_air.filter("a.id = 'JFK'").select("a.id", "c.id").orderBy("c.id").show()
# print(two_air.count(), "paths of exactly 2 flights;", air.find("...").count(), "of 1 or 2 flights")

## 7. Subgraphs

**Slides**: *Subgraph (1): filters*, *Subgraph (2): from a motif*

* `filterVertices(cond)` keeps the vertices that satisfy the condition **and drops the edges that lost an endpoint**.
* `filterEdges(cond)` keeps the edges that satisfy the condition, and **all** the vertices.
* `dropIsolatedVertices()` removes the vertices that are in no edge.
* All three return a GraphFrame, so they compose. And any DataFrame with columns `src`, `dst` is a valid edges table: for instance the edges of a motif result (`select("e.*")`).

In [ ]:
# Goal: keep users older than 30 and their friend relationships only
g1 = (g.filterVertices("age > 30")
       .filterEdges("relationship = 'friend'")
       .dropIsolatedVertices())
g1.vertices.show()
g1.edges.show()

### Your turn 7

**Goal**: keep the airports with `pax >= 40` and the flights shorter than 1,000 km, then drop the isolated airports. Show the remaining airports and print their number and the number of flights.

*Hint*: `air.filterVertices("pax >= 40").filterEdges("km < 1000").dropIsolatedVertices()`, then `.vertices`, `.edges`, `count()`.

*Expected*: 5 airports (`BOS`, `CDG`, `FRA`, `JFK`, `LHR`) and 5 flights. `LIS`, `KIX` and `PPT` are removed by `pax >= 40`. `MAD` is dropped because its only remaining flight (to Paris, 1,050 km) is too long, `ORD` because all its flights are longer than 1,000 km, and `NRT` because its only partner `KIX` is gone.

In [ ]:
# YOUR CODE HERE
# sub = (air.filterVertices(...)
#           .filterEdges(...)
#           .dropIsolatedVertices())
# sub.vertices.orderBy("id").show()
# print(sub.vertices.count(), sub.edges.count())

**Slides**: *Subgraph (2): from a motif*

A motif result is a DataFrame: selecting its edge column (`e.*`) gives a new edges table, hence a **new graph**. Here, the follow edges from a younger to an older user.

In [ ]:
# Goal: a subgraph from a motif: follow edges from a younger to an older user
paths = (g.find("(a)-[e]->(b)")
          .filter("e.relationship = 'follow'")
          .filter("a.age < b.age"))
e2 = paths.select("e.*")
g2 = GraphFrame(g.vertices, e2)
g2.edges.show()

### Your turn 8

**Goal**: build the subgraph of the flights that have a **return flight** (`a` to `b` and `b` to `a`), with the motif `(a)-[e]->(b); (b)-[e2]->(a)` and `select("e.*")`. Show its edges. How many vertices does the new graph have, and how many remain after `dropIsolatedVertices()`?

*Hint*: Same steps as the slide: `find(...)`, `select("e.*")`, `GraphFrame(air.vertices, edges)`, `.edges`, `.dropIsolatedVertices().vertices.count()`.

*Expected*: 4 flights: `CDG`-`LHR` and `KIX`-`NRT`, each in both directions. The graph keeps its 11 vertices, 4 remain after `dropIsolatedVertices()`.

In [ ]:
# YOUR CODE HERE
# both = air.find("...").select("e.*")
# air_rt = GraphFrame(air.vertices, both)
# air_rt.edges.orderBy("src", "dst").show()
# print(air_rt.vertices.count(), "->", air_rt.dropIsolatedVertices().vertices.count())

## 8. Built-in algorithms

**Slides**: *Built-in algorithms: one call, one new column*

Every algorithm is a **method of the GraphFrame**. The result is the vertices DataFrame plus **one new column** (for `pageRank`, a GraphFrame: vertices + `pagerank`, edges + `weight`).
Every result is a DataFrame: filter it, join it, sort it, save it.

The next sections take one algorithm each, in the order of the slides: PageRank, connected components, strongly connected components, shortest paths, BFS, triangle count, label propagation.

### 8.1 PageRank

**Slides**: *PageRank*

`pageRank(resetProbability, maxIter | tol)`: the rank of a vertex is high when many vertices of high rank point to it. `resetProbability` is the probability to jump to a random vertex (0.15 in the paper). Set either `maxIter` (fixed number of rounds) or `tol` (stop when the change is below the tolerance).

In [ ]:
# Goal: who is the most influential user?
results = g.pageRank(resetProbability=0.15, tol=0.01)
ranks = results.vertices.select("id", F.round("pagerank", 3).alias("pagerank"))
ranks.orderBy(F.desc("pagerank")).show()
results.edges.select("src", "dst", "weight").orderBy("src", "dst").show()

### Your turn 9

**Goal**: run PageRank on the airline network (`resetProbability=0.15`, `tol=0.01`) and print the rank of all airports, highest first (3 decimals, ties broken by `id`). Which airports have the lowest rank? Which airport has the highest in-degree?

*Hint*: `air.pageRank(resetProbability=0.15, tol=0.01)`, then `.vertices.select("id", F.round("pagerank", 3).alias("pagerank"))`, `orderBy(F.desc("pagerank"), "id")`. In-degree: `air.inDegrees`.

*Expected*: The top 3 are `CDG` 1.916, `LHR` 1.801, `JFK` 1.696. The lowest are `LIS` and `PPT` (0.171): nobody flies to them. `CDG` also has the highest in-degree (3).

In [ ]:
# YOUR CODE HERE
# res = air.pageRank(resetProbability=0.15, tol=0.01)
# res.vertices.select("id", F.round("pagerank", 3).alias("pagerank")).orderBy(...).show(11)
# air.inDegrees.orderBy(...).show(1)

### 8.2 Connected components

**Slides**: *Connected components*

`connectedComponents()`: vertices + `component` (an arbitrary long id). Two vertices have the same `component` when a path links them, **ignoring the direction** of the edges.
The algorithm is iterative: Spark needs a **checkpoint directory** (or `use_local_checkpoints=True`).

In [ ]:
# Goal: which users are linked, ignoring edge direction?
# required before connectedComponents (or use_local_checkpoints=True)
spark.sparkContext.setCheckpointDir("/tmp/gf-checkpoints")
cc = g.connectedComponents()
members = F.sort_array(F.collect_list("id")).alias("members")
cc.groupBy("component").agg(members).show(truncate=False)

### Your turn 10

**Goal**: how many connected components has the airline network? List the members of each, sorted (`F.sort_array(F.collect_list("id"))`).

*Hint*: `air.connectedComponents()`, then `groupBy("component").agg(F.sort_array(F.collect_list("id")).alias("members"))`, `orderBy("members")`.

*Expected*: 3 components: `[BOS, CDG, FRA, JFK, LHR, LIS, MAD, ORD]` (Europe and the US), `[KIX, NRT]` (Japan) and `[PPT]` alone. The component ids can differ from yours.

In [ ]:
# YOUR CODE HERE
# members = F.sort_array(F.collect_list("id")).alias("members")
# cc_air = air.connectedComponents()
# cc_air.groupBy(...).agg(members).orderBy("members").show(truncate=False)

### 8.3 Strongly connected components

**Slides**: *Strongly connected components*

`stronglyConnectedComponents(maxIter)`: vertices + `component`. Two vertices are in the same group when each one can **reach the other following the direction** of the edges. The groups are a finer partition than the connected components.

In [ ]:
# Goal: groups of users who can all reach each other following edge direction
scc = g.stronglyConnectedComponents(maxIter=10)
members = F.sort_array(F.collect_list("id")).alias("members")
scc.groupBy("component").agg(members).orderBy("members").show(truncate=False)

### Your turn 11

**Goal**: list the strongly connected components of the airline network (`maxIter=10`), as on the slide. Why are `MAD` and `LIS` not in the big group, although they are in the same connected component?

*Hint*: `air.stronglyConnectedComponents(maxIter=10)`, then the same `groupBy` / `members` as in the previous exercise.

*Expected*: 5 groups: `[BOS, CDG, FRA, JFK, LHR, ORD]`, `[KIX, NRT]`, `[LIS]`, `[MAD]`, `[PPT]`. `MAD` flies to Paris, but no flight comes back to Madrid except from Lisbon, and nobody flies to Lisbon: no cycle goes through them.

In [ ]:
# YOUR CODE HERE
# scc_air = air.stronglyConnectedComponents(maxIter=...)
# scc_air.groupBy(...).agg(members).orderBy("members").show(truncate=False)

### 8.4 Shortest paths

**Slides**: *Shortest paths to landmarks*

`shortestPaths(landmarks, is_directed=True)`: vertices + `distances`, a **map** landmark -> number of hops (the edges are not weighted). A landmark that cannot be reached is **absent** from the map: `{}` = no landmark reachable.

In [ ]:
# Goal: how many hops from each user to Alice and to David?
sp = g.shortestPaths(landmarks=["a", "d"])
sp.select("id", "distances").orderBy("id").show(truncate=False)

### Your turn 12

**Goal**: compute the number of flights from each airport to New York (`JFK`) and to Madrid (`MAD`). How many flights from Lisbon to New York? Is there a route from New York to Madrid?

*Hint*: `air.shortestPaths(landmarks=["JFK", "MAD"])`, then `select("id", "distances").orderBy("id")`.

*Expected*: `LIS` -> `{JFK -> 4, MAD -> 1}`: 4 flights from Lisbon to New York. The row of `JFK` is `{JFK -> 0}`: `MAD` is not in the map, so there is no route from New York to Madrid.

In [ ]:
# YOUR CODE HERE
# sp_air = air.shortestPaths(landmarks=[...])
# sp_air.select("id", "distances").orderBy("id").show(truncate=False)

### 8.5 Breadth-first search

**Slides**: *BFS: paths from Esther on follow edges*

`bfs(fromExpr, toExpr, edgeFilter, maxPathLength)`: the **shortest paths** from the vertices that satisfy `fromExpr` to the vertices that satisfy `toExpr`. One row per path, with the columns `from`, `e0`, `v1`, `e1`, ..., `to`: the number of columns depends on the length of the path. `edgeFilter` excludes edges from the search.

In [ ]:
# Goal: paths from Esther to users younger than 32, without using friend edges
paths = g.bfs("name = 'Esther'", "age < 32",
              edgeFilter="relationship != 'friend'", maxPathLength=3)
paths.show(truncate=False)

### Your turn 13

**Goal**: find the shortest route from Madrid (`id = 'MAD'`) to an airport of the United States (`country = 'US'`). Then search again **without using the flights of British Airways** (`edgeFilter="airline != 'BA'"`) and print the number of paths found.

*Hint*: `air.bfs("id = 'MAD'", "country = 'US'", maxPathLength=5)`; `.count()` for the second search.

*Expected*: One path of 3 flights: `MAD` -> `CDG` -> `LHR` -> `JFK`. Without British Airways: 0 paths. The only flight from Europe to the US is `LHR` -> `JFK` (`BA`).

In [ ]:
# YOUR CODE HERE
# air.bfs("...", "...", maxPathLength=5).show(truncate=False)
# n = air.bfs("...", "...", edgeFilter="...", maxPathLength=5).count()
# print("paths without BA:", n)

### 8.6 Triangle count

**Slides**: *Triangle count*

`triangleCount(storage_level)`: vertices + `count`, the number of **triangles** (three vertices all linked) that contain the vertex. The direction of the edges is ignored. In GraphFrames 0.12.3 the Python method needs its argument (`StorageLevel.MEMORY_AND_DISK`): `g.triangleCount()` raises a `TypeError`.

In [ ]:
# Goal: how many triangles (3 users all linked) go through each user?
from pyspark import StorageLevel
tc = g.triangleCount(StorageLevel.MEMORY_AND_DISK)
tc.select("id", "count").orderBy("id").show()

### Your turn 14

**Goal**: count the triangles through each airport, highest first (ties broken by `id`). How many triangles are there in the whole network?

*Hint*: `air.triangleCount(StorageLevel.MEMORY_AND_DISK)`. Each triangle is counted once for each of its 3 vertices: what do you do with the sum of the counts?

*Expected*: `CDG`, `JFK`, `LHR` have 2 triangles; `BOS`, `FRA`, `ORD` have 1; the others 0. In total: `3 triangles` (the sum of the counts is 9, divided by 3).

In [ ]:
# YOUR CODE HERE
# tc_air = air.triangleCount(...)
# tc_air.select("id", "count").orderBy(...).show()
# print(..., "triangles")

### 8.7 Label propagation

**Slides**: *Label propagation (LPA): communities*

`labelPropagation(maxIter)`: vertices + `label` (a community id). Each vertex starts with its own label; at each round it adopts the **most frequent label among its neighbors** (messages flow in both directions). Vertices that share a label form a community. No parameter such as the number of communities.

In [ ]:
# Goal: detect communities without any parameter
lpa = g.labelPropagation(maxIter=5)
lpa.select("id", "label").orderBy("label", "id").show()

### Your turn 15 (bonus)

**Goal**: run label propagation on the airline network (`maxIter=5`). How many communities (distinct labels)? Check that a label is **never shared by two connected components**: for each label, count the distinct components of its airports (from `connectedComponents`), and print the maximum.

*Hint*: `air.labelPropagation(maxIter=5)`; `join` with `air.connectedComponents()` on `id`; `groupBy("label").agg(F.count_distinct("component"))`; `F.max`.

*Expected*: The maximum is `1`: labels travel along edges, so they never cross two components. The number of communities depends on the Spark settings (we got 6).

In [ ]:
# YOUR CODE HERE
# lpa_air = air.labelPropagation(maxIter=5)
# cc2 = air.connectedComponents()
# joined = lpa_air.select("id", "label").join(cc2.select("id", "component"), "id")
# per_label = joined.groupBy("label").agg(F.count_distinct("component").alias("n_components"))
# print(lpa_air.select("label").distinct().count(), "communities; max components per label:", ...)

## 9. aggregateMessages

**Slides**: *aggregateMessages: one round of messages*, *aggregateMessages (1): sending*, *aggregateMessages (2): aggregating*

One round of messages, in two phases (the first two phases of a Pregel superstep):
1. **Send**: for every edge, build a message for its `src` and/or its `dst` from the columns of the triplet: `AM.src["age"]`, `AM.dst["age"]`, `AM.edge["col"]`.
2. **Aggregate**: each vertex aggregates the messages it received with a Spark aggregate on `AM.msg` (`F.sum`, `F.max`, `F.collect_list`...).

The result has **one row per vertex that received a message**: Gabby receives nothing, so she has no row. Here each user sums the ages of its neighbors, in both directions.

In [ ]:
# Goal: sum of the ages of each user's neighbors (both directions)
from graphframes.lib import AggregateMessages as AM
agg = g.aggregateMessages(
    F.sum(AM.msg).alias("summedAges"),
    sendToSrc=AM.dst["age"],
    sendToDst=AM.src["age"])
agg.orderBy("id").show()

Bob counts Charlie **twice** (94 = 30 + 30 + 34): `b` and `c` are linked by two edges, so Bob receives two messages from Charlie. Here are the messages received by Esther (`e`): one per incident edge, with the ages of the two endpoints.

In [ ]:
# Goal: the messages received by Esther (e), one per incident edge
g.triplets.filter("src.id = 'e' or dst.id = 'e'").select("src.id", "dst.id", "src.age", "dst.age").show()

### Your turn 16

**Goal**: with `aggregateMessages`, compute for each airport the **total kilometers of the flights that arrive** there. Show the result, highest first (ties broken by `id`), and print how many airports received a message.

*Hint*: A message only for the destination of each edge: `sendToDst=AM.edge["km"]`, aggregated by `F.sum(AM.msg).alias("km_in")`.

*Expected*: `CDG` 7224, `JFK` 6730, `ORD` 1380, `LHR` 981, `MAD` 500, `FRA` 479, `KIX` 400, `NRT` 400, `BOS` 300. `9 airports received a message`: `LIS` and `PPT` have no incoming flight, so no row.

In [ ]:
# YOUR CODE HERE
# km_in = air.aggregateMessages(F.sum(AM.msg).alias("km_in"), sendToDst=...)
# km_in.orderBy(...).show()
# print(km_in.count(), "airports received a message")

## 10. The Pregel API

**Slides**: *The Pregel API: four calls*, *PageRank as a Pregel program*

`g.pregel` builds an **iterative** BSP computation whose vertex state is a set of DataFrame columns. Each round = **send** a message along the edges, **aggregate** the messages of each vertex, **update** the state of the vertex:

* `withVertexColumn(name, initial_expr, update_expr)`: the state column, its initial value, and its new value from the aggregated message `Pregel.msg()` (**update**)
* `sendMsgToDst(expr)` (and `sendMsgToSrc`): the message of each edge, built from `Pregel.src("col")`, `Pregel.dst("col")`, `Pregel.edge("col")` (**send**)
* `aggMsgs(agg_expr)`: how the messages of a vertex are combined, e.g. `F.sum(Pregel.msg())` (**aggregate**)
* `setMaxIter(n)` and `run()`: the number of rounds, then the final vertices DataFrame

**PageRank as a Pregel program** (the pseudo-code of the Pregel paper): at superstep 0, `rank = 1 / N`. At each following superstep, `rank = 0.15 / N + 0.85 * sum(incoming messages)`, and every vertex sends `rank / outDegree` along each of its out-edges.

**Slides**: *PageRank with the Pregel API*

The same program with the Pregel API. `coalesce` gives 0 to a vertex that received no message (Gabby). The vertices also need their `outDegree`: Gabby has none, so `fillna(0)`.

In [ ]:
# Goal: PageRank by hand with the Pregel API
# rank = alpha/N + (1 - alpha) * sum of rank(u)/outDegree(u)
from graphframes.lib import Pregel
alpha, N = 0.15, g.vertices.count()          # N = 7
v2 = (g.vertices.join(g.outDegrees, "id", "left")
      .fillna(0, ["outDegree"]))             # Gabby: 0
g2 = GraphFrame(v2, g.edges)
ranks = (g2.pregel
    .setMaxIter(5)
    .withVertexColumn("rank", F.lit(1.0 / N),
        F.coalesce(Pregel.msg(), F.lit(0.0)) * (1 - alpha)
        + alpha / N)
    .sendMsgToDst(Pregel.src("rank") / Pregel.src("outDegree"))
    .aggMsgs(F.sum(Pregel.msg()))
    .run())
result = ranks.select("id", "outDegree",
                      F.round("rank", 3).alias("rank"))
result.orderBy(F.desc("rank")).show()

**Slides**: *Pregel PageRank: round 0 and round 1, send*, *Pregel PageRank: round 1, aggregate and update*

The slides show the values on the graph, round by round. Here is the same computation in three cells, with the values of the slides (`0.143` = 1/7 for every vertex at round 0).

In [ ]:
# Goal: round 0 (initial state) and round 1, send: one message per edge
# round 0: initial state
r0 = g2.vertices.withColumn("rank", F.lit(1.0 / N))
r0.select("id", "outDegree", F.round("rank", 3).alias("rank")).orderBy("id").show()
# round 1, send: one message per edge = rank(src) / outDegree(src)
msgs = (GraphFrame(r0, g.edges).triplets
        .select(F.col("src.id").alias("src"), F.col("dst.id").alias("dst"),
                F.round(F.col("src.rank") / F.col("src.outDegree"), 3).alias("msg")))
msgs.orderBy("src", "dst").show()

In [ ]:
# Goal: round 1, aggregate: sum of the messages per destination
msgs.groupBy("dst").agg(F.round(F.sum("msg"), 3).alias("sum_msg")).orderBy("dst").show()

In [ ]:
# Goal: round 1, update: the Pregel API after one round (the three phases in one call)
r1 = (g2.pregel.setMaxIter(1)
      .withVertexColumn("rank", F.lit(1.0 / N),
                        F.coalesce(Pregel.msg(), F.lit(0.0)) * (1 - alpha) + alpha / N)
      .sendMsgToDst(Pregel.src("rank") / Pregel.src("outDegree"))
      .aggMsgs(F.sum(Pregel.msg())).run())
r1.select("id", "outDegree", F.round("rank", 3).alias("rank")).orderBy("id").show()
print(r1.agg(F.round(F.sum("rank"), 3)).collect()[0][0])

Charlie receives from Bob and Fanny: 0.143 + 0.143 = 0.286, so his rank is 0.286 x 0.85 + 0.15 / 7 = 0.264. Gabby gets no message: 0.15 / 7 = 0.021. The ranks now sum to **0.879**: Gabby's mass is lost (she has no out-edge).

**Slides**: *After 5 rounds: Pregel vs. g.pageRank*

In [ ]:
# Goal: compare the hand-written Pregel PageRank (previous slides) with g.pageRank
ranks.select("id", F.round("rank", 3).alias("rank")).orderBy(F.desc("rank")).show()

5 rounds are not enough: `b` and `c` swap places at every round (they form a 2-cycle `b` <-> `c`). Convergence takes about 20 rounds (`b` 0.335, `c` 0.339). The built-in `g.pageRank` gives the same ranking, up to a normalization: its ranks sum to N = 7, ours to 0.879.

In [ ]:
# Goal: 20 rounds of the Pregel program, then the built-in g.pageRank
r20 = (g2.pregel.setMaxIter(20)
       .withVertexColumn("rank", F.lit(1.0 / N),
                         F.coalesce(Pregel.msg(), F.lit(0.0)) * (1 - alpha) + alpha / N)
       .sendMsgToDst(Pregel.src("rank") / Pregel.src("outDegree"))
       .aggMsgs(F.sum(Pregel.msg())).run())
r20.select("id", F.round("rank", 3).alias("rank")).orderBy(F.desc("rank")).show()
pr10 = g.pageRank(resetProbability=0.15, maxIter=20)
pr10.vertices.select("id", F.round("pagerank", 3).alias("pagerank")).orderBy(F.desc("pagerank")).show()

### Your turn 17

**Goal**: write PageRank with the Pregel API on the airline network, as on the slide: `alpha = 0.15`, **10 rounds**. The vertices need their `outDegree` (`PPT` has none: `fillna`). Print `id`, `outDegree` and the rank (3 decimals) of all airports, highest first (ties broken by `id`), then the sum of the ranks.

*Hint*: Same four calls as the slide: `withVertexColumn("rank", F.lit(1.0 / N), ...)`, `sendMsgToDst(Pregel.src("rank") / Pregel.src("outDegree"))`, `aggMsgs(F.sum(Pregel.msg()))`, `setMaxIter(10)`. N = `air.vertices.count()` = 11.

*Expected*: The top 3: `CDG` 0.159, `LHR` 0.152, `JFK` 0.144. `LIS` and `PPT` both get `0.15 / 11` = 0.014: `PPT` has no out-edge, `LIS` receives nothing. The sum is `0.923` (`PPT` loses its mass).

In [ ]:
# YOUR CODE HERE
# alpha, N = 0.15, air.vertices.count()
# v_pr = air.vertices.join(air.outDegrees, "id", "left").fillna(0, ["outDegree"])
# air_pr = GraphFrame(v_pr, air.edges)
# pr = (air_pr.pregel
#       .setMaxIter(...)
#       .withVertexColumn("rank", ..., ...)
#       .sendMsgToDst(...)
#       .aggMsgs(...)
#       .run())
# pr.select("id", "outDegree", F.round("rank", 3).alias("rank")).orderBy(...).show(11)
# print(pr.agg(F.round(F.sum("rank"), 3)).collect()[0][0])

## 11. Challenge (not in the slides): reconnect Lisbon and Madrid

In the airline network, Lisbon and Madrid cannot be reached from the big European and US group (section *Strongly connected components*). The airline wants to **add one flight** so that `LIS` and `MAD` join the big strongly connected group.

**Questions**
1. *Think first, then check.* Add the flight `CDG` -> `LIS` (`AF`, 1450 km). Build the new graph `air2` (a GraphFrame is **immutable**: build a new edges DataFrame with `union`, then a new GraphFrame). How many strongly connected components are there now, and who is in the big one? Has the number of **connected** components changed?
2. Try the flight `MAD` -> `LIS` (`TP`, 500 km) instead, in a graph `air3`. How many strongly connected components now? Explain the difference with question 1.
3. *Bonus.* Run `pageRank` on `air` and on `air2`: what is the rank of `LIS` before and after the new flight (3 decimals)?

*Hint*: `air.edges.union(spark.createDataFrame([(...)], air.edges.columns))`, then `GraphFrame(air.vertices, new_edges)`. The code of the exercises *Strongly connected components*, *Connected components* and *PageRank* can be reused.

*Expected*:
1. 3 strongly connected components: `[BOS, CDG, FRA, JFK, LHR, LIS, MAD, ORD]`, `[KIX, NRT]`, `[PPT]`. There were 5. The number of connected components stays 3: the new flight links two airports that were already in the same component.
2. 4 strongly connected components: `[BOS, CDG, FRA, JFK, LHR, ORD]`, `[KIX, NRT]`, `[LIS, MAD]`, `[PPT]`. `LIS` and `MAD` now fly to each other, but nobody in the big group can reach them: a group needs a cycle that goes **into** it and back out. In question 1, `CDG` -> `LIS` -> `MAD` -> `CDG` is such a cycle.
3. `LIS` goes from 0.171 to 0.741.

In [ ]:
# YOUR CODE HERE
# members = F.sort_array(F.collect_list("id")).alias("members")
# --- question 1
# new_flight = spark.createDataFrame([("CDG", "LIS", "AF", 1450)], air.edges.columns)
# air2 = GraphFrame(air.vertices, air.edges.union(new_flight))
# air2.stronglyConnectedComponents(maxIter=10).groupBy(...).agg(members).orderBy("members").show(truncate=False)
# print(air2.connectedComponents().select("component").distinct().count(), "connected components")
# --- question 2
# air3 = ...
# --- question 3 (bonus)
# for name, gr in [("before", air), ("after", air2)]:
#     ...

## Key takeaways

* A GraphFrame = **two DataFrames**: vertices (`id`) and edges (`src`, `dst`). Everything you know about DataFrames still works on them.
* **Queries**: `find` (motifs) matches a pattern, `filterVertices` / `filterEdges` / `dropIsolatedVertices` build subgraphs.
* **Algorithms**: one method per algorithm, one new column in the result: `pageRank`, `connectedComponents`, `stronglyConnectedComponents`, `shortestPaths`, `bfs`, `triangleCount`, `labelPropagation`.
* **Your own algorithms**: `aggregateMessages` = one round (send, aggregate); the Pregel API = many rounds (send, aggregate, update).
* A vertex with no edge has no row in degrees or in `aggregateMessages`: join back with `g.vertices`.

Stop Spark at the end of the session:

In [ ]:
spark.stop()